In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

from msd.graph import MSDGraph
from msd.plotter import MSDPlotter
from utils import HypergraphPlotter, remove_mergeid_suffixes

from hypergraph.api.lib.tools import RGL
import clr
clr.AddReference("System.Collections")
from System import Collections
from RGeoLib import Vec3d, NLine, GeometryUtils, DataNodeUtils

# Picking a random graph to work with

In [ ]:
# Load training data directly (without process_data)
data_dir = Path("../../dataset/msd/modified-swiss-dwellings-v2/train/graph_out")
training_files = list(data_dir.glob("*.pickle"))
print(f"Found {len(training_files)} training files")

# Load a specific file for testing
file_name = 3566
test_file = None
for file_path in training_files:
    if file_path.stem == str(file_name):
        test_file = file_path
        break

if test_file:
    print(f"Loading file: {test_file.name}")
    with open(test_file, 'rb') as f:
        graph_data = pickle.load(f)
    print(f"Graph loaded: {graph_data.number_of_nodes()} nodes, {graph_data.number_of_edges()} edges")
else:
    # Fallback to first file if specific file not found
    test_file = training_files[0]
    print(f"File {file_name} not found, using: {test_file.name}")
    with open(test_file, 'rb') as f:
        graph_data = pickle.load(f)


# Visualizing the graph 

In [ ]:
# import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

plotter = MSDPlotter()
# Create a figure with two subplots side by side
fig, ax1 = plt.subplots(1, 1, figsize=(10, 10))

plotter.plot_floor(graph_data, ax1, 
           node_size=50,                    # Size of nodes
           edge_size=2)                     # Width of edges



def add_room_legend(ax1, graph_data):

    # Create legend for room types present in the graph
    room_types_in_graph = [graph_data.nodes[node]['room_type'] for node in graph_data.nodes()]
    room_type_counts = Counter(room_types_in_graph)

    # Create legend patches for room types that exist in the graph
    legend_patches = []
    for room_type_index in sorted(room_type_counts.keys()):
        room_name = plotter.ROOM_NAMES[room_type_index]
        color = plotter.CMAP_ROOMTYPE(room_type_index)
        count = room_type_counts[room_type_index]
        
        patch = mpatches.Patch(color=color, label=f'{room_name} ({count})')
        legend_patches.append(patch)

    # Add legend to the left plot
    legend = ax1.legend(handles=legend_patches, 
                        title='Room Types (count)', 
                        loc='center left', 
                        bbox_to_anchor=(1.02, 0.5),
                        fontsize=10,
                        title_fontsize=12)
    return legend

# Create legend for edge types (connectivity)
edge_legend_patches = [
    mpatches.Patch(color='black', label='Door/Passage'),
    mpatches.Patch(color='red', label='Entrance')
]

# Add legends
legend1 = add_room_legend(ax1, graph_data)
legend2 = ax1.legend(handles=edge_legend_patches, 
                    title='Edge Types', 
                    loc='center left', 
                    bbox_to_anchor=(1.02, 0.15),
                    fontsize=10,
                    title_fontsize=12)

# Add both legends to the left plot
ax1.add_artist(legend1)  # Need to add the first legend back since the second one replaces it

ax1.set_title(f"Floor Plan Graph\nNodes: {graph_data.number_of_nodes()}, Edges: {graph_data.number_of_edges()}")
ax1.axis('equal')

# Dividing the graph into subgraphs and plotting a subgraph

In [ ]:
# Divide graph into subgraphs using MSDGraph class
subgraphs = MSDGraph.extract_subgraphs_without_entrance(graph_data)
# Calculate convex hull for first subgraph using MSDGraph class
target_subgraph = subgraphs[2]

fig, ax1 = plt.subplots(1, 1, figsize=(10, 10))

plotter.plot_floor(target_subgraph, ax1, 
           node_size=50,                    # Size of nodes
           edge_size=2)

# Add legends
legend1 = add_room_legend(ax1, target_subgraph)
ax1.axis('equal')



# Rotating a subgraph

In [ ]:
rotated_subgraph = MSDGraph.rotate_x_aligned(target_subgraph)
fig, ax1 = plt.subplots(1, 1, figsize=(10, 10))

plotter.plot_floor(rotated_subgraph, ax1, 
           node_size=50,                    # Size of nodes
           edge_size=2)

# Add legends
legend1 = add_room_legend(ax1, target_subgraph)
ax1.axis('equal')

# Converting a subgraph to hypergraph

In [ ]:
hull_data = MSDGraph.get_geometry_based_boundaries(rotated_subgraph)
face = MSDGraph.convex_hull_to_face_wrapper(hull_data['convex_hull_geometry'])
nmesh, room_faces = MSDGraph.create_nmesh_from_subgraph(rotated_subgraph)
connectivity_dict, _ = MSDGraph.build_connectivity_dict(rotated_subgraph)
# data_node = DataNodeUtils.CreateDataNodeWithConnectivity(nmesh, face, connectivity_dict)
data_node = DataNodeUtils.CreateDataNodeWithConnectivity(
    nmesh, 
    face, 
    connectivity_dict,
)

# Plotting the hypergraph

In [ ]:
plotter = HypergraphPlotter()
data_node_text = RGL.DataNode.serializeDataNode(data_node)
data_node_text = remove_mergeid_suffixes(data_node_text)
plotter.plot_hypergraph(data_node_text)

# Defining the boundary polygon and generating floorplan

In [ ]:
p1 = RGL.Vec3d(x=0, y=0, z=0)
p2 = RGL.Vec3d(x=0, y=8, z=0)
p3 = RGL.Vec3d(x=10, y=8, z=0)
p4 = RGL.Vec3d(x=10, y=3, z=0)
p5 = RGL.Vec3d(x=12, y=3, z=0)
p6 = RGL.Vec3d(x=12, y=0, z=0)

points_list = Collections.Generic.List[RGL.Vec3d]()
for p in [p1, p2, p3, p4, p5, p6]:
    points_list.Add(p)

face_bounds = RGL.NFace(points_list)
nface_list = Collections.Generic.List[RGL.NFace]()
nface_list.Add(face_bounds)
boundary_mesh = RGL.NMesh(nface_list)

data_node = RGL.DataNode.deserializeDataNode(data_node_text)
generated_mesh = RGL.DataNode.subdivideWithMeshRatio(data_node, boundary_mesh)
# generated_mesh = RGL.DataNode.subdivideWithMeshRatio(data_node, boundary_mesh)


In [ ]:
plotter.simple_plot_generated_mesh(generated_mesh)